In [ ]:
# Production-Ready RAGAS Evaluation System for RAG Agent
# Comprehensive evaluation with robust error handling, logging, and metrics

import os
import json
import time
import logging
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import userdata
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision,
    answer_correctness,
    answer_similarity
)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage

In [ ]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


class ProductionRAGASEvaluator:
    """
    Production-ready RAGAS evaluator with comprehensive metrics,
    error handling, retry logic, and detailed reporting.
    """

    def __init__(
        self,
        app,
        tools,
        output_dir: str = "ragas_evaluations",
        model: str = "gpt-4o-mini",
        max_retries: int = 3,
        retry_delay: int = 2
    ):
        """
        Initialize the RAGAS evaluator.

        Args:
            app: Your compiled LangGraph agent
            tools: List of tools for the agent
            output_dir: Directory to save evaluation results
            model: OpenAI model for evaluation
            max_retries: Maximum retries for failed requests
            retry_delay: Delay between retries in seconds
        """
        self.app = app
        self.tools = tools
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        self.model = model
        self.max_retries = max_retries
        self.retry_delay = retry_delay

        # Initialize OpenAI clients
        self._setup_openai_clients()

    def _setup_openai_clients(self):
        """Setup OpenAI LLM and embeddings with proper configuration"""
        try:
            api_key = userdata.get('OPENAI_API_KEY')
            if not api_key:
                raise ValueError("OPENAI_API_KEY not found in Colab secrets")

            os.environ['OPENAI_API_KEY'] = api_key

            self.llm = ChatOpenAI(
                model=self.model,
                api_key=api_key,
                temperature=0,
                request_timeout=60
            )

            self.embeddings = OpenAIEmbeddings(
                model="text-embedding-3-small",
                api_key=api_key
            )

            logger.info(f"✅ OpenAI clients initialized with model: {self.model}")

        except Exception as e:
            logger.error(f"❌ Failed to setup OpenAI clients: {e}")
            raise

    def create_production_test_dataset(self) -> List[Dict[str, str]]:
        """
        Create a comprehensive test dataset covering various question types.

        Production tip: Expand this to 20-50 questions covering:
        - Factual questions
        - Multi-hop reasoning
        - Comparison questions
        - Definition questions
        - Edge cases
        """

        test_data = [
            # Factual recall
            {
                "question": "What are the two main types of memory in LLM-powered autonomous agents according to Lilian Weng?",
                "ground_truth": "According to Lilian Weng, LLM-powered autonomous agents have two main types of memory: (1) Short-term memory, which utilizes in-context learning and includes information that fits in the current context window, and (2) Long-term memory, which provides agents with the capability to retain and recall information over extended periods using external vector stores and fast retrieval mechanisms.",
                "category": "factual"
            },
            # Process explanation
            {
                "question": "How does task decomposition work in agent planning?",
                "ground_truth": "Task decomposition in agent planning involves breaking down large, complex tasks into smaller, manageable subgoals. This enables efficient handling of complex tasks by allowing the agent to tackle each subgoal systematically, making the overall problem more tractable and improving the quality of results.",
                "category": "process"
            },
            # Comparison
            {
                "question": "How does Tree of Thoughts differ from Chain of Thought?",
                "ground_truth": "Tree of Thoughts extends Chain of Thought by enabling deliberate problem solving through exploring multiple reasoning paths simultaneously, like a search tree. Instead of following a single chain of reasoning, it explores different branches of thought and can backtrack or explore alternative solutions.",
                "category": "comparison"
            },
            # Definition/Classification
            {
                "question": "What are AutoGPT, GPT-Engineer, and BabyAGI?",
                "ground_truth": "AutoGPT, GPT-Engineer, and BabyAGI are proof-of-concept demos that serve as inspiring examples of LLM-powered autonomous agents. They demonstrate the potential of using large language models as core controllers for autonomous systems that can break down tasks, use tools, and work toward goals with minimal human intervention.",
                "category": "definition"
            },
            # Component identification
            {
                "question": "What are the key components of an LLM-powered autonomous agent?",
                "ground_truth": "The key components of an LLM-powered autonomous agent include: (1) Planning - task decomposition and self-reflection, (2) Memory - both short-term and long-term memory systems, and (3) Tool use - the ability to call external APIs and use tools to gather information or take actions.",
                "category": "components"
            },
            # Multi-hop reasoning
            {
                "question": "How do ReAct and Reflexion approaches improve agent performance?",
                "ground_truth": "ReAct improves agent performance by combining reasoning and acting, allowing the agent to generate reasoning traces and task-specific actions in an interleaved manner. Reflexion enhances this by adding self-reflection capabilities, enabling the agent to learn from past mistakes and improve its decision-making over time through feedback loops.",
                "category": "multi-hop"
            }
        ]

        logger.info(f"📝 Created test dataset with {len(test_data)} questions")
        return test_data

    def run_agent_with_retry(
        self,
        question: str,
        retry_count: int = 0
    ) -> Tuple[str, List[str]]:
        """
        Run agent with retry logic for robustness.

        Returns:
            Tuple of (answer, contexts)
        """
        try:
            inputs = {
                "messages": [HumanMessage(content=question)],
                "tools": self.tools
            }

            answer = ""
            retrieved_contexts = []

            for output in self.app.stream(inputs):
                for key, value in output.items():
                    if key == "retrieve" and "messages" in value:
                        context = value["messages"][-1].content
                        retrieved_contexts.append(context)
                    elif key == "generate" and "messages" in value:
                        answer = value["messages"][-1].content

            # Process contexts
            contexts = []
            if retrieved_contexts:
                full_context = "\n".join(retrieved_contexts)
                context_chunks = [
                    chunk.strip()
                    for chunk in full_context.split("\n\n")
                    if chunk.strip()
                ]
                contexts = context_chunks[:10]  # Top 10 contexts

            return answer, contexts if contexts else ["No context retrieved"]

        except Exception as e:
            if retry_count < self.max_retries:
                logger.warning(f"⚠️ Retry {retry_count + 1}/{self.max_retries} for question")
                time.sleep(self.retry_delay * (retry_count + 1))
                return self.run_agent_with_retry(question, retry_count + 1)
            else:
                logger.error(f"❌ Failed after {self.max_retries} retries: {e}")
                return f"Error: {str(e)}", ["Error occurred"]

    def collect_evaluation_data(
        self,
        test_data: List[Dict[str, str]]
    ) -> List[Dict[str, Any]]:
        """
        Collect evaluation data by running agent on test questions.

        Returns:
            List of evaluation results with metadata
        """
        results = []

        logger.info(f"🚀 Starting data collection for {len(test_data)} questions")

        for i, item in enumerate(test_data, 1):
            question = item["question"]
            ground_truth = item["ground_truth"]
            category = item.get("category", "general")

            logger.info(f"\n📝 Question {i}/{len(test_data)} [{category}]")
            logger.info(f"❓ {question[:100]}...")

            start_time = time.time()
            answer, contexts = self.run_agent_with_retry(question)
            execution_time = time.time() - start_time

            result = {
                "question": question,
                "answer": answer,
                "contexts": contexts,
                "ground_truth": ground_truth,
                "category": category,
                "execution_time": execution_time,
                "answer_length": len(answer.split()),
                "num_contexts": len(contexts),
                "timestamp": datetime.now().isoformat()
            }

            results.append(result)

            logger.info(f"✅ Answer: {len(answer.split())} words | "
                       f"Contexts: {len(contexts)} | "
                       f"Time: {execution_time:.2f}s")

            # Rate limiting
            time.sleep(1)

        return results

    def run_ragas_evaluation(
        self,
        results_data: List[Dict[str, Any]],
        metrics_subset: Optional[List] = None
    ) -> Optional[Any]:
        """
        Run RAGAS evaluation with comprehensive metrics.

        Args:
            results_data: List of evaluation results
            metrics_subset: Optional subset of metrics to use

        Returns:
            RAGAS evaluation results or None if failed
        """
        logger.info("\n🔍 Running RAGAS evaluation...")

        # Prepare dataset
        dataset_dict = {
            "question": [item["question"] for item in results_data],
            "answer": [item["answer"] for item in results_data],
            "contexts": [item["contexts"] for item in results_data],
            "ground_truth": [item["ground_truth"] for item in results_data]
        }

        dataset = Dataset.from_dict(dataset_dict)

        # Select metrics
        if metrics_subset is None:
            metrics_to_use = [
                answer_relevancy,
                faithfulness,
                context_recall,
                context_precision,
                answer_correctness,
                answer_similarity
            ]
        else:
            metrics_to_use = metrics_subset

        logger.info(f"📊 Using {len(metrics_to_use)} metrics")

        try:
            result = evaluate(
                dataset,
                metrics=metrics_to_use,
                llm=self.llm,
                embeddings=self.embeddings,
                raise_exceptions=False
            )

            logger.info("✅ RAGAS evaluation completed")
            return result

        except Exception as e:
            logger.error(f"❌ RAGAS evaluation failed: {e}")
            import traceback
            logger.error(traceback.format_exc())
            return None

    def generate_comprehensive_report(
        self,
        results_data: List[Dict[str, Any]],
        ragas_result: Any
    ) -> str:
        """Generate detailed evaluation report with insights"""

        report = []
        report.append("=" * 80)
        report.append("📊 PRODUCTION RAGAS EVALUATION REPORT")
        report.append("=" * 80)
        report.append(f"Evaluation Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        report.append(f"Model: {self.model}")
        report.append(f"Total Questions: {len(results_data)}")

        # Overall RAGAS scores
        if ragas_result is not None:
            report.append("\n" + "=" * 80)
            report.append("📈 RAGAS METRICS SCORES")
            report.append("=" * 80)

            try:
                if hasattr(ragas_result, 'to_pandas'):
                    df = ragas_result.to_pandas()
                    mean_scores = df.mean()

                    for metric in mean_scores.index:
                        score = mean_scores[metric]
                        if not pd.isna(score):
                            emoji = "🟢" if score > 0.8 else "🟡" if score > 0.6 else "🔴"
                            report.append(f"{emoji} {metric.replace('_', ' ').title()}: {score:.4f}")

                    # Category breakdown
                    report.append("\n" + "=" * 80)
                    report.append("📂 PERFORMANCE BY CATEGORY")
                    report.append("=" * 80)

                    # Only compute category stats if we have numeric columns
                    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

                    if numeric_cols:
                        df_with_categories = df[numeric_cols].copy()
                        df_with_categories['category'] = [item['category'] for item in results_data]

                        category_means = df_with_categories.groupby('category').mean(numeric_only=True)
                        for category in category_means.index:
                            report.append(f"\n{category.upper()}:")
                            for metric in category_means.columns:
                                score = category_means.loc[category, metric]
                                if not pd.isna(score):
                                    report.append(f"  • {metric}: {score:.4f}")

            except Exception as e:
                report.append(f"Error parsing RAGAS results: {e}")

        # Performance statistics
        report.append("\n" + "=" * 80)
        report.append("⚡ PERFORMANCE STATISTICS")
        report.append("=" * 80)

        exec_times = [item['execution_time'] for item in results_data]
        answer_lengths = [item['answer_length'] for item in results_data]
        context_counts = [item['num_contexts'] for item in results_data]

        report.append(f"Average Execution Time: {sum(exec_times)/len(exec_times):.2f}s")
        report.append(f"Average Answer Length: {sum(answer_lengths)/len(answer_lengths):.1f} words")
        report.append(f"Average Contexts Retrieved: {sum(context_counts)/len(context_counts):.1f}")

        # Recommendations
        report.append("\n" + "=" * 80)
        report.append("💡 RECOMMENDATIONS")
        report.append("=" * 80)

        if ragas_result is not None and hasattr(ragas_result, 'to_pandas'):
            try:
                df = ragas_result.to_pandas()
                numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

                if numeric_cols:
                    mean_scores = df[numeric_cols].mean()

                    if 'faithfulness' in mean_scores and mean_scores['faithfulness'] < 0.7:
                        report.append("⚠️ Low faithfulness score - Review retrieval quality and prompt engineering")

                    if 'context_recall' in mean_scores and mean_scores['context_recall'] < 0.7:
                        report.append("⚠️ Low context recall - Consider improving retrieval strategy")

                    if 'answer_relevancy' in mean_scores and mean_scores['answer_relevancy'] < 0.7:
                        report.append("⚠️ Low answer relevancy - Review generation prompts")

                    if 'context_precision' in mean_scores and mean_scores['context_precision'] < 0.7:
                        report.append("⚠️ Low context precision - Retrieved contexts may not be relevant enough")

                    # If all metrics are good
                    if all(mean_scores[m] >= 0.7 for m in mean_scores.index):
                        report.append("✅ All metrics are performing well! Continue monitoring.")
                else:
                    report.append("⚠️ No numeric metrics available for recommendations")
            except Exception as e:
                report.append(f"⚠️ Could not generate recommendations: {e}")

        report.append("\n" + "=" * 80)
        report.append("📖 METRIC DEFINITIONS")
        report.append("=" * 80)
        report.append("• Answer Relevancy: How well the answer addresses the question")
        report.append("• Faithfulness: How well the answer is grounded in retrieved context")
        report.append("• Context Recall: Coverage of ground truth by retrieved contexts")
        report.append("• Context Precision: Relevance of retrieved contexts")
        report.append("• Answer Correctness: Semantic similarity to ground truth")
        report.append("• Answer Similarity: Embedding-based similarity to ground truth")

        return "\n".join(report)

    def save_results(
        self,
        results_data: List[Dict[str, Any]],
        ragas_result: Any,
        report: str
    ) -> str:
        """Save all evaluation results with timestamp"""

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        # Save raw data
        results_df = pd.DataFrame(results_data)
        raw_path = self.output_dir / f"raw_data_{timestamp}.csv"
        results_df.to_csv(raw_path, index=False)
        logger.info(f"💾 Raw data saved: {raw_path}")

        # Save RAGAS scores
        if ragas_result is not None:
            try:
                if hasattr(ragas_result, 'to_pandas'):
                    ragas_df = ragas_result.to_pandas()
                    scores_path = self.output_dir / f"ragas_scores_{timestamp}.csv"
                    ragas_df.to_csv(scores_path, index=False)
                    logger.info(f"💾 RAGAS scores saved: {scores_path}")
            except Exception as e:
                logger.warning(f"⚠️ Could not save RAGAS scores: {e}")

        # Save report
        report_path = self.output_dir / f"report_{timestamp}.txt"
        with open(report_path, 'w') as f:
            f.write(report)
        logger.info(f"💾 Report saved: {report_path}")

        # Save metadata
        metadata = {
            "timestamp": timestamp,
            "model": self.model,
            "num_questions": len(results_data),
            "output_dir": str(self.output_dir)
        }
        metadata_path = self.output_dir / f"metadata_{timestamp}.json"
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=2)

        return timestamp

    def run_production_evaluation(
        self,
        use_full_metrics: bool = True
    ) -> Tuple[List[Dict[str, Any]], Any]:
        """
        Run complete production-level RAGAS evaluation.

        Args:
            use_full_metrics: Whether to use all metrics (slower but comprehensive)

        Returns:
            Tuple of (results_data, ragas_scores)
        """
        logger.info("🎯 Starting Production RAGAS Evaluation")
        logger.info(f"⚙️ Configuration: Full metrics={use_full_metrics}")

        # Create test dataset
        test_data = self.create_production_test_dataset()

        # Collect evaluation data
        results_data = self.collect_evaluation_data(test_data)

        # Run RAGAS evaluation
        metrics = None if use_full_metrics else [
            answer_relevancy,
            faithfulness,
            context_recall
        ]
        ragas_result = self.run_ragas_evaluation(results_data, metrics)

        # Generate report
        report = self.generate_comprehensive_report(results_data, ragas_result)
        print("\n" + report)

        # Save results
        timestamp = self.save_results(results_data, ragas_result, report)

        logger.info(f"\n✅ Production evaluation complete!")
        logger.info(f"📁 Results saved in: {self.output_dir}")

        return results_data, ragas_result


# ============================================
# USAGE EXAMPLES
# ============================================
"""
# Basic usage with default settings
evaluator = ProductionRAGASEvaluator(app=app, tools=tools)
results, scores = evaluator.run_production_evaluation()

# Advanced usage with custom configuration
evaluator = ProductionRAGASEvaluator(
    app=app,
    tools=tools,
    output_dir="my_evaluations",
    model="gpt-4o",  # Use GPT-4 for more accurate evaluation
    max_retries=5
)
results, scores = evaluator.run_production_evaluation(use_full_metrics=True)

# Quick evaluation with core metrics only
results, scores = evaluator.run_production_evaluation(use_full_metrics=False)
"""